# 00.3 — The MATLAB → Python mental model

MATLAB and Python feel similar at a glance — both are interactive, both have a REPL, both have a numeric stack. But the **mental models** behind them differ in several places, and the gaps explain most of the early MATLAB-to-Python frustrations.

This notebook covers the five biggest shifts:

1. **Everything is an object** — including numbers, functions, modules, types themselves.
2. **Indices start at 0, ranges are half-open** — `1:n` means something different.
3. **Indentation is syntax** — Python has no `end` keyword.
4. **Mutable vs immutable** — assignment and passing arguments behave differently than MATLAB.
5. **Namespaces and modules** — there is no MATLAB path; explicit `import` everywhere.

Each section has runnable code so you can poke at the differences yourself.

## Section 1 — What MATLAB does

In MATLAB, a typical session feels like this:

```matlab
% Everything goes in the workspace.
x = 5;
y = [1, 2, 3];

% Functions live in their own .m files; the file name = the function name.
% Calling a function copies its arguments (pass-by-value semantics by default).
z = myFunc(x);   % x is unchanged inside myFunc

% Arrays use 1-indexed inclusive ranges.
y(1)             % first element
y(1:end)         % all elements
y(2:end-1)       % middle elements
```

Python overlaps with this in concept but differs in mechanics. Let's see how.

## Section 2 — The Python concepts you need

### 2.1 — Everything is an object

In Python, every value is an **object** with methods you can call. Even integers.

In [ ]:
x = 5
type(x)      # <class 'int'> — even integers are class instances

In [ ]:
x.bit_length()    # int has methods you can call
# 5 in binary is 101, which is 3 bits long

In [ ]:
# Functions are objects too. You can pass them as arguments, store them in lists.
def square(n):
    return n * n

ops = [square, abs, len]
ops[0](7)    # calls square(7)

**Why this matters:** when you see code like `optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)`, every piece is an object. `model.parameters()` returns an iterator object; `Adam(...)` constructs an optimizer object; `lr=1e-3` is a keyword argument. There's no special syntax — it's all calling methods and constructing objects.

Compare to MATLAB's `trainNetwork(layers, options)` which feels similar but hides the object model behind a wrapper function.

### 2.2 — Indices start at 0, ranges are half-open

Python's indexing convention catches every MATLAB user at least once. The trap is that `range(1, 5)` does NOT include 5.

In [ ]:
y = [10, 20, 30, 40, 50]
y[0]      # first element — MATLAB: y(1)

In [ ]:
y[-1]     # last element — MATLAB: y(end)
# Negative indices count from the end. y[-2] is the second-to-last.

In [ ]:
y[1:4]    # elements at indices 1, 2, 3 (NOT 4)
# MATLAB analog: y(2:4) — both endpoints INclusive
# Python ranges are HALF-open: start INclusive, stop EXclusive

In [ ]:
list(range(1, 5))    # the integers from 1 up to (not including) 5

**Practical implication:** when you port `for kidx = 1:NumEpochs` from MATLAB, the Python equivalent is `for k in range(num_epochs):`. The loop variable `k` runs from `0` to `num_epochs - 1`. If anything inside the loop body uses `k` to index a 1-indexed external system (a SLURM array task, a MATLAB-side log line), remember to add `1`.

Numpy arrays inherit Python's 0-indexing. Use `arr[0]` for the first element, `arr[-1]` for the last, `arr[1:4]` for indices 1–3.

### 2.3 — Indentation is syntax

MATLAB uses `end` to close blocks. Python uses **indentation** — the amount of whitespace at the start of each line is what tells the parser which lines belong to which block.

In [ ]:
# Python
for i in range(3):
    print(f"i = {i}")          # 4 spaces of indent → inside the loop
    if i == 1:
        print("  middle")        # 8 spaces → inside the if AND the loop
    # back to 4 spaces → out of the if, still in the loop
print("done")                    # 0 indent → out of the loop

Equivalent MATLAB:

```matlab
for i = 0:2
    fprintf('i = %d\n', i);
    if i == 1
        fprintf('  middle\n');
    end
end
fprintf('done\n');
```

**Practical implications:**

* Always use **4 spaces** per indent level (this is the universal Python convention, codified in PEP 8). Most editors insert these when you press Tab.
* If you mix tabs and spaces, Python raises `IndentationError`. Configure your editor to insert spaces for `.py` files.
* There is no `end` keyword. A block ends when the indentation drops.
* This means you can't write multi-line `if` statements as one-liners with `;` separators — each statement needs its own line.

### 2.4 — Mutable vs immutable

MATLAB passes arguments by value: when you call `myFunc(x)`, the function gets a COPY of `x`. Modifying it inside `myFunc` does not change the caller's `x`.

Python passes a *reference to the object* into the function ("call by object sharing"), so **whether the function can modify the caller's data depends on whether the object is mutable**.

* **Immutable**: `int`, `float`, `str`, `tuple` — assignment inside the function rebinds the local name, never the caller's.
* **Mutable**: `list`, `dict`, `set`, `numpy.ndarray`, `torch.Tensor` — methods like `.append()`, `[i] = ...`, `+=` modify the same object the caller is holding.

In [ ]:
# Immutable — looks like MATLAB pass-by-value.
def add_one(n):
    n = n + 1      # rebinds the LOCAL n, doesn't touch the caller
    return n

x = 5
y = add_one(x)
print(f"x = {x}, y = {y}")    # x still 5, y is 6

In [ ]:
# Mutable — modifying a list inside a function modifies the caller's list too.
def append_one(lst):
    lst.append(1)    # modifies the SAME object

my_list = [10, 20]
append_one(my_list)
print(my_list)       # [10, 20, 1] — the caller sees the modification

Worth a picture. The diagram below shows what's happening in memory for the two cases above — the immutable case rebinds a local name to a NEW object, while the mutable case has two names pointing at the SAME object.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax_imm, ax_mut) = plt.subplots(1, 2, figsize=(11, 4))

def draw_name(ax, x, y, name, color="#dddddd"):
    rect = mpatches.FancyBboxPatch((x - 0.45, y - 0.18), 0.9, 0.36,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x, y, name, ha="center", va="center", fontsize=14, family="monospace")

def draw_value(ax, x, y, value, color="#cce4ff"):
    rect = mpatches.FancyBboxPatch((x - 0.55, y - 0.25), 1.1, 0.5,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x, y, value, ha="center", va="center", fontsize=13, family="monospace")

def arrow(ax, x1, y1, x2, y2, color="black", style="->"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle=style, color=color, lw=1.5))

# Immutable (int) case
ax_imm.set_title("Immutable — int\n`x = 5; y = x; y = y + 1`", fontsize=12)
draw_name(ax_imm, 1.0, 3.5, "x")
draw_name(ax_imm, 3.0, 3.5, "y")
draw_value(ax_imm, 1.0, 2.0, "5", color="#cce4ff")
draw_value(ax_imm, 3.0, 0.5, "6 (new!)", color="#ffe4cc")
arrow(ax_imm, 1.0, 3.2, 1.0, 2.4)
arrow(ax_imm, 3.0, 3.2, 3.0, 0.9, color="#aa0000")
ax_imm.text(2.0, 2.0, "x still\npoints here", ha="center", va="center", fontsize=10, color="#666")
ax_imm.set_xlim(-0.2, 4.2); ax_imm.set_ylim(-0.2, 4.2)
ax_imm.set_aspect("equal"); ax_imm.axis("off")

# Mutable (list) case
ax_mut.set_title("Mutable — list\n`lst = [1]; lst2 = lst; lst2.append(2)`", fontsize=12)
draw_name(ax_mut, 1.0, 3.5, "lst")
draw_name(ax_mut, 3.0, 3.5, "lst2")
draw_value(ax_mut, 2.0, 2.0, "[1, 2]", color="#cce4ff")
arrow(ax_mut, 1.0, 3.2, 1.8, 2.3)
arrow(ax_mut, 3.0, 3.2, 2.2, 2.3)
ax_mut.text(2.0, 0.5, "BOTH names point\nto the SAME list", ha="center", va="center",
    fontsize=10, color="#aa0000", style="italic")
ax_mut.set_xlim(-0.2, 4.2); ax_mut.set_ylim(-0.2, 4.2)
ax_mut.set_aspect("equal"); ax_mut.axis("off")

plt.tight_layout()
plt.show()
print("Left:  int is immutable — `y = y + 1` rebinds y to a NEW object.")
print("Right: list is mutable — `lst2.append(2)` mutates the SHARED object.")

**Practical implication:** when a function takes a tensor or numpy array, **operations that don't reassign typically modify in-place**:

* `tensor += 1` modifies in-place — the caller's tensor changes.
* `tensor = tensor + 1` rebinds the local name to a new tensor — the caller's tensor does NOT change.

This is the source of an entire class of bugs that don't exist in MATLAB. PyTorch even has a convention: tensor methods ending in `_` (like `tensor.zero_()`, `tensor.add_(1)`) modify in-place; methods without the trailing underscore return a new tensor.

### 2.5 — Namespaces and modules

MATLAB has a single global namespace per workspace. Adding a directory to the path makes every `.m` file in it callable from anywhere. This is convenient but doesn't scale — name collisions and accidentally shadowing built-in functions are common.

Python is the opposite: **nothing is callable until you explicitly import it**.

In [ ]:
# Import the whole module under a short alias.
import numpy as np

np.arange(5)    # access via the alias

In [ ]:
# Import a specific name from a module.
from pathlib import Path

Path.cwd()    # no module prefix; Path is in the local namespace

In [ ]:
# Import a deeply-nested name from this project.
from neural_data_decoding.sweeps import total_sweep_count, lookup

print(f"Sweep table has {total_sweep_count()} entries.")
print(f"Entry 1: {lookup(1).description}")

Once you import a name, it lives in your **module's namespace**. Other modules that want it need their own import statement. There is no MATLAB-style "add to path and it's everywhere" — every Python file is its own namespace.

This is a feature, not a bug:

* You can have two libraries that both define a `sum()` function with no collision.
* Reading the top of a `.py` file tells you exactly what external code it uses.
* Type checkers (pyright) and IDEs can resolve every name back to its source.

## Section 3 — The `neural_data_decoding` implementation

Look at the top of a real production file to see all five concepts in action:

In [ ]:
# Read the first 50 lines of the CLI source
from pathlib import Path
src = Path("../../src/neural_data_decoding/cli.py").read_text().splitlines()[:50]
for i, line in enumerate(src, start=1):
    print(f"{i:3} | {line}")

Things to notice:

* **Top of the excerpt:** every external dependency (`argparse`, `json`, `sys`, `pathlib`, `torch`, `omegaconf`) is explicitly imported. There is no implicit path.
* **Lower half of the excerpt:** project-internal modules are imported with leading dots (`.data.dataset`, `.interop`). The dot tells Python to look inside the current package.
* **`from .interop import (...)`:** the parentheses just let the import list span multiple lines — idiomatic Python for long import lists.
* The CLI's `main()` function (further down) loops with `for ...:` — no `end` keyword anywhere; indentation marks each block.

## Section 4 — Hands-on exercises

### Exercise 4.1 — half-open ranges

Without running the code, predict what `list(range(0, 10, 2))` returns. Then check below.

In [ ]:
# Reveal:
list(range(0, 10, 2))

### Exercise 4.2 — mutable surprise

What does this print?

In [ ]:
def add_default(item, into=[]):
    into.append(item)
    return into

print(add_default(1))
print(add_default(2))
print(add_default(3))

Each call adds to the SAME list — because the default `[]` is created once when the function is defined, not once per call. This is the famous **mutable default argument** trap. The fix:

```python
def add_default(item, into=None):
    if into is None:
        into = []
    into.append(item)
    return into
```

**This is the #1 most common bug a MATLAB user writes in their first month of Python.** Now you've seen it; you won't write it.

### Exercise 4.3 — explicit imports

In a fresh notebook cell try `Path.cwd()`. It will fail. Why?

(Because you haven't imported `Path`. Every notebook is a separate namespace, and imports from one notebook do NOT carry into another.)

## Section 5 — Common errors

### `IndentationError: unexpected indent`

Your editor's auto-indent put in a different amount of whitespace than the surrounding code. (Mixing tabs and spaces raises its own, more specific `TabError`.) Replace tabs with 4 spaces.

### `NameError: name 'x' is not defined`

Either you misspelled the name, or you used it before assigning it, or it lives in another module and you forgot to import it. Read the traceback — it tells you which file and line.

### `IndexError: list index out of range`

You used `arr[len(arr)]` thinking it would give you the last element. Use `arr[-1]` or `arr[len(arr) - 1]`.

### Mutable default argument bug (from exercise 4.2)

The function holds onto state between calls. Switch to `None` and create the mutable object inside the function body.

### "My import works at the top of the file but fails inside a function"

It doesn't — Python re-runs the import statement (cheaply, from a cache) any time it's encountered. The actual issue is usually a **circular import** (file A imports from B which imports from A). Refactor so the import is one-directional.

### `AttributeError: 'NoneType' object has no attribute 'x'`

A function returned `None` (Python's `null`) and you tried to use it as an object. In MATLAB this would manifest as `Subscripted assignment between dissimilar structures` or similar. The traceback line tells you which expression was `None`.

## Section 6 — Further reading

* [PEP 8 — Python style guide](https://peps.python.org/pep-0008/). The 20-minute read that gets you to writing idiomatic Python.
* [PEP 20 — The Zen of Python](https://peps.python.org/pep-0020/). 19 aphorisms that explain Python's design philosophy. Run `import this` in any Python REPL.
* ["Common gotchas" in the Python docs](https://docs.python-guide.org/writing/gotchas/). Mutable defaults, late binding, more.
* `notebooks/01_python_for_matlab_users/` — the next module covers Python syntax in detail, with MATLAB analogs for each construct.

Next up: **[00.4 IDE deep dive](00.4_ide_deep_dive.ipynb)** — the foolproof editor-setup guide — then the project-craft trio ([00.5 command line](00.5_the_command_line_for_matlab_users.ipynb), [00.6 git](00.6_git_and_github_for_matlab_users.ipynb), [00.7 packaging](00.7_pip_packaging_and_project_anatomy.ipynb)) and the [00.8 capstone](00.8_build_a_dl_project_from_scratch.ipynb). Welcome to Python.